In [ ]:
import os
import logging
from dotenv import load_dotenv
from telegram import Update, ReplyKeyboardMarkup, ReplyKeyboardRemove
from telegram.ext import (
    ApplicationBuilder, 
    CommandHandler, 
    MessageHandler, 
    filters, 
    ContextTypes, 
    ConversationHandler
)
from fitnesslogic import chat_with_bmi

# Load environment variables
load_dotenv()
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO)

# Define conversation states
WEIGHT, HEIGHT, GENDER, AGE = range(4)

# --- CONVERSATION HANDLERS (The Setup Wizard) ---

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Starts the conversation and asks for weight."""
    await update.message.reply_text(
        "👋 Hello! I'm your fitness assistant.\n\n"
        "To give you the best advice, I need to set up your profile.\n"
        "First, please enter your **Weight in kg** (e.g., 70):",
        parse_mode="Markdown"
    )
    return WEIGHT

async def receive_weight(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Stores weight and asks for height."""
    try:
        weight = float(update.message.text)
        context.user_data['weight'] = weight
        await update.message.reply_text("Great. Now, enter your **Height in cm** (e.g., 175):", parse_mode="Markdown")
        return HEIGHT
    except ValueError:
        await update.message.reply_text("⚠️ Please enter a valid number for weight (e.g., 70.5). Try again:")
        return WEIGHT

async def receive_height(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Stores height and asks for gender."""
    try:
        height = float(update.message.text)
        context.user_data['height'] = height
        
        # Create buttons for gender selection
        reply_keyboard = [["Male", "Female"]]
        await update.message.reply_text(
            "Got it. What is your **Gender**?", 
            reply_markup=ReplyKeyboardMarkup(reply_keyboard, one_time_keyboard=True, resize_keyboard=True),
            parse_mode="Markdown"
        )
        return GENDER
    except ValueError:
        await update.message.reply_text("⚠️ Please enter a valid number for height (e.g., 170). Try again:")
        return HEIGHT

async def receive_gender(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Stores gender and asks for age."""
    gender = update.message.text
    if gender not in ["Male", "Female"]:
        await update.message.reply_text("Please tap one of the buttons (Male or Female).")
        return GENDER
    
    context.user_data['gender'] = gender
    await update.message.reply_text("Almost done! Enter your **Age**:", reply_markup=ReplyKeyboardRemove(), parse_mode="Markdown")
    return AGE

async def receive_age(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Stores age and ends the setup."""
    try:
        age = int(update.message.text)
        context.user_data['age'] = age
        
        # Summary
        w = context.user_data['weight']
        h = context.user_data['height']
        g = context.user_data['gender']
        
        await update.message.reply_text(
            f"✅ **Profile Saved!**\n\nStats: {g}, {age}y/o, {w}kg, {h}cm.\n\n"
            "You can now ask me anything about workouts or nutrition!",
            parse_mode="Markdown"
        )
        return ConversationHandler.END
    except ValueError:
        await update.message.reply_text("⚠️ Please enter a valid whole number for age (e.g., 25). Try again:")
        return AGE

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Cancels and ends the conversation."""
    await update.message.reply_text("Setup canceled. Type /start to try again.", reply_markup=ReplyKeyboardRemove())
    return ConversationHandler.END

# --- CHAT LOGIC (The Actual Bot) ---

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handles normal chat messages after profile is set."""
    user_text = update.message.text

    # Check if user has completed the setup
    if 'weight' not in context.user_data:
        await update.message.reply_text("⚠️ I don't know your stats yet. Please tap /start to set up your profile.")
        return

    await context.bot.send_chat_action(chat_id=update.effective_chat.id, action="typing")

    # Retrieve stored data
    weight = context.user_data['weight']
    height = context.user_data['height']
    gender = context.user_data['gender']
    age = context.user_data['age']

    # Get history (Optional: You can implement simple history storage here if needed)
    chat_history = [] 

    response = chat_with_bmi(
        question=user_text,
        weight=weight,
        height=height,
        gender=gender,
        age=age,
        chat_history=chat_history
    )

    try:
        if len(response) > 4096:
            for x in range(0, len(response), 4096):
                await update.message.reply_text(response[x:x+4096])
        else:
            await update.message.reply_text(response)
    except Exception as e:
        print(f"❌ FAILED TO SEND MESSAGE: {e}")
        await update.message.reply_text("Error sending response. Try a simpler question.")

# --- MAIN ---

def main():
    if TELEGRAM_BOT_TOKEN is None:
        raise ValueError("TELEGRAM_BOT_TOKEN is missing! Check your .env file.")

    app = ApplicationBuilder().token(TELEGRAM_BOT_TOKEN).build()

    # Create the conversation handler
    conv_handler = ConversationHandler(
        entry_points=[CommandHandler("start", start)],
        states={
            WEIGHT: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_weight)],
            HEIGHT: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_height)],
            GENDER: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_gender)],
            AGE: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_age)],
        },
        fallbacks=[CommandHandler("cancel", cancel)],
    )

    # Add handlers
    app.add_handler(conv_handler) # Handles /start and the setup flow
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message)) # Handles regular chat

    print("Bot is running...")
    app.run_polling()

if __name__ == "__main__":
    main()